# Analysis tools

Now that we [know how to access data with `NanoEvents`](https://github.com/iris-hep/us-atlas-idap-training-2024/tree/main/PHYSLITE), let's go through some useful columnar analysis tools and idioms for building collections of results, namely, the eventual output of a `coffea` callable (or processor).
The most familiar type of output may be the histogram (one type of accumulator).

We'll just look at single files for the time being to keep things simple.

## Rapid review of what we've already seen

In [ ]:
from pathlib import Path
import warnings

from matplotlib import pyplot as plt
import awkward as ak
from hist import Hist
from coffea.nanoevents import NanoEventsFactory, PHYSLITESchema
from coffea.analysis_tools import PackedSelection
import mplhep

PHYSLITESchema.warn_missing_crossrefs = False

In [ ]:
from importlib.metadata import version

for package in ["numpy", "awkward", "uproot", "coffea"]:
    print(f"# {package}: v{version(package)}")

In [ ]:
# HZZ -> 4l sample

# local
# file_path = '/Users/iason/DAOD_PHYSLITE.38191712._000001.pool.root.1'

# stream
# file_path = "root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000001.pool.root.1"

# XCache
file_path = "root://xcache.af.uchicago.edu:1094//root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000001.pool.root.1"

There is lots of information in the files, but for this example we're only going to look at a few fields:
* Event information
* Electrons
* Muons
* Jets
* B-tagging

In [ ]:
def filter_name(name):
    """
    Load only the properties/variables needed.
    """
    return name in (
        "EventInfoAuxDyn.mcEventWeights",
        "EventInfoAuxDyn.mcChannelNumber",
        #
        "AnalysisElectronsAuxDyn.pt",
        "AnalysisElectronsAuxDyn.eta",
        "AnalysisElectronsAuxDyn.phi",
        "AnalysisElectronsAuxDyn.m",
        "AnalysisElectronsAuxDyn.DFCommonElectronsLHLoose",
        "AnalysisElectronsAuxDyn.charge",
        #
        "AnalysisMuonsAuxDyn.pt",
        "AnalysisMuonsAuxDyn.eta",
        "AnalysisMuonsAuxDyn.phi",
        "AnalysisMuonsAuxDyn.m",
        "AnalysisMuonsAuxDyn.charge",
        "AnalysisMuonsAuxDyn.quality",
        #
        "AnalysisJetsAuxDyn.pt",
        "AnalysisJetsAuxDyn.eta",
        "AnalysisJetsAuxDyn.phi",
        "AnalysisJetsAuxDyn.m",
        #
        "BTagging_AntiKt4EMPFlowAuxDyn.DL1dv01_pb",
        "BTagging_AntiKt4EMPFlowAuxDyn.DL1dv01_pc",
        "BTagging_AntiKt4EMPFlowAuxDyn.DL1dv01_pu",
    )

There will be some warnings from `coffea`, but in this case they can be ignored.

In [ ]:
warnings.filterwarnings(
    "ignore",
    message="Skipping ",
    category=UserWarning,
)

# coffea 2026.7 reads into *virtual arrays* by default (mode="virtual"): each
# branch stays on disk and is materialized only when first used. There is no
# dask task graph and no .compute() step -- results are produced eagerly as the
# arrays are touched. We pass mode explicitly here just to make that visible.
events = NanoEventsFactory.from_root(
    {file_path: "CollectionTree"},
    schemaclass=PHYSLITESchema,
    mode="virtual",
    iteritems_options=dict(filter_name=filter_name),
).events()

and we get the fields we requested

In [ ]:
events.fields

and the subfields that were requested for each field

In [ ]:
for _field in events.fields:
    print(f"* {_field}: {events[_field].fields}")

## `PackedSelection`

This class can store several boolean arrays in a memory-efficient mannner and evaluate arbitrary combinations of boolean requirements in an CPU-efficient way. Supported inputs include 1D `numpy` or `awkward` arrays. This makes it a good tool to form analysis signal and control regions, and to implement cutflow or "N-1" plots.

Below we create a packed selection with some typical selections for a $Z$+jets study, to be used later to form same-sign and opposite-sign $ee$ and $\mu\mu$ event categories/regions.

We'll use [ATLAS open data electroweak boson simulation](https://opendata.cern.ch/record/80010) for this ( DOI:[10.7483/OPENDATA.ATLAS.K5SU.X65Y](http://doi.org/10.7483/OPENDATA.ATLAS.K5SU.X65Y)). Specifically `mc20_13TeV_MC_Sh_2211_Zee_maxHTpTV2_CVetoBVeto` and `mc20_13TeV_MC_Sh_2211_Zmumu_maxHTpTV2_CVetoBVeto` samples.

In [ ]:
# Zee

# local
# file_path = "/Users/iason/DAOD_PHYSLITE.37621317._000001.pool.root.1"

# stream
# file_path = "root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.37621317._000001.pool.root.1"

# XCache
file_path = "root://xcache.af.uchicago.edu:1094//root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.37621317._000001.pool.root.1"

In [ ]:
events = NanoEventsFactory.from_root(
    {file_path: "CollectionTree"},
    schemaclass=PHYSLITESchema,
    mode="virtual",
    iteritems_options=dict(filter_name=filter_name),
).events()

In [ ]:
selection = PackedSelection()

selection.add("two_electrons", ak.num(events.Electrons, axis=1) == 2)
selection.add("electrons_opposite_sign", ak.sum(events.Electrons.charge, axis=1) == 0)
selection.add("no_electrons", ak.num(events.Electrons, axis=1) == 0)

selection.add("two_muons", ak.num(events.Muons, axis=1) == 2)
selection.add("muons_opposite_sign", ak.sum(events.Muons.charge, axis=1) == 0)
selection.add("no_muons", ak.num(events.Muons, axis=1) == 0)


selection.add(
    "lead_pt_20",
    # assuming one of `two_electrons` or `two_muons` is imposed, this implies at least one is above threshold
    ak.any(events.Electrons.pt >= 20.0, axis=1)
    | ak.any(events.Muons.pt >= 20.0, axis=1),
)

print(selection.names)

To evaluate a boolean mask (e.g. to filter events) we can use the `selection.all(*names)` function, which will compute the AND of all listed boolean selections

In [ ]:
selection.all("two_electrons", "no_muons", "lead_pt_20")

We can also be more specific and require that a specific set of selections have a given value (with the unspecified ones allowed to be either `True` or `False`) using `selection.require`

In [ ]:
selection.require(two_electrons=True, no_muons=True, electrons_opposite_sign=False)

Using the Python syntax for passing an arguments variable, we can easily implement a "N-1" style selection

In [ ]:
all_cuts = {"two_electrons", "no_muons", "lead_pt_20"}
results = {}
for cut in all_cuts:
    n_events = ak.sum(selection.all(*(all_cuts - {cut})), axis=0)
    results[cut] = n_events

results["None"] = ak.sum(selection.all(*all_cuts), axis=0)

for cut, n_events in results.items():
    print(f"Events passing all cuts, ignoring '{cut}': {n_events}")

Luckily coffea implements that for you. And also a "cutflow" selection via `selection.cutflow`

In [ ]:
nminusone = selection.nminusone("two_electrons", "no_muons", "lead_pt_20")
nminusone.print()

In [ ]:
h, labels = nminusone.yieldhist()
h.plot()
plt.xticks(plt.gca().get_xticks(), labels, rotation=45)
plt.show()

## Bringing it together

Let's build a callable function that books a few results:
* the sum of weights for the events processed (to use for later luminosity-normalizing the yields)
* a histogram of the dilepton invariant mass, with category axes for various selection regions of interest

In [ ]:
events = NanoEventsFactory.from_root(
    {file_path: "CollectionTree"},
    schemaclass=PHYSLITESchema,
    mode="virtual",
    iteritems_options=dict(filter_name=filter_name),
).events()

In [ ]:
events

In [ ]:
def results(events):
    selection = PackedSelection()

    selection.add("two_electrons", ak.num(events.Electrons, axis=1) == 2)
    selection.add(
        "electrons_opposite_sign", ak.sum(events.Electrons.charge, axis=1) == 0
    )
    selection.add("no_electrons", ak.num(events.Electrons, axis=1) == 0)

    selection.add("two_muons", ak.num(events.Muons, axis=1) == 2)
    selection.add("muons_opposite_sign", ak.sum(events.Muons.charge, axis=1) == 0)
    selection.add("no_muons", ak.num(events.Muons, axis=1) == 0)

    selection.add(
        "lead_pt_20",
        # assuming one of `two_electrons` or `two_muons` is imposed, this implies at least one is above threshold
        ak.any(events.Electrons.pt >= 20.0, axis=1)
        | ak.any(events.Muons.pt >= 20.0, axis=1),
    )

    regions = {
        "ee": {
            "two_electrons": True,
            "no_muons": True,
            "lead_pt_20": True,
            "electrons_opposite_sign": True,
        },
        "ee_same_sign": {
            "two_electrons": True,
            "no_muons": True,
            "lead_pt_20": True,
            "electrons_opposite_sign": False,
        },
        "mumu": {
            "two_muons": True,
            "no_electrons": True,
            "lead_pt_20": True,
            "muons_opposite_sign": True,
        },
        "mumu_same_sign": {
            "two_muons": True,
            "no_electrons": True,
            "lead_pt_20": True,
            "muons_opposite_sign": False,
        },
    }

    mass_hist = (
        Hist.new.StrCat(regions.keys(), name="region")
        .Reg(60, 60, 120, name="mass", label="$m_{ll}$ [GeV]")
        .Weight()
    )

    for region, cuts in regions.items():
        good_event = selection.require(**cuts)

        if region.startswith("ee"):
            leptons = events.Electrons[good_event]
        elif region.startswith("mumu"):
            # Hack for the time being given PHYSLITESchema needs fixing
            _muons = events.Muons[good_event]
            _muons["m"] = ak.zeros_like(_muons.pt)
            leptons = _muons
        lep1 = leptons[:, 0]
        lep2 = leptons[:, 1]
        mass = (lep1 + lep2).mass

        mass_hist.fill(
            region=region,
            mass=mass,
        )

    out = {
        "sumw": ak.sum(events.EventInfo.mcEventWeights, axis=0),
        "mass": mass_hist,
    }

    return out

So when we run `results(events)` we get back a plain `dict` of already-filled results (a sum of weights and a `hist` histogram). With the virtual-array backend the histograms are filled eagerly as the arrays are materialized — there is no lazy dask task graph and nothing to `.compute()`.

In [ ]:
output = results(events)

output

Thanks to `hist` we can slo see nice Jupyter [`reprs`](https://docs.python.org/3/library/functions.html#repr) of the objects

In [ ]:
output["mass"]

In [ ]:
output["mass"][sum, :]

In [ ]:
plot_dir = Path().cwd() / "plots"
plot_dir.mkdir(exist_ok=True)

In [ ]:
mplhep.style.use(mplhep.style.ATLAS)

fig, ax = plt.subplots()

output["mass"][sum, :].plot1d(ax=ax, label="$ll$ mass")
ax.legend()

fig.savefig(plot_dir / "ll_mass.png")

In [ ]:
fig, ax = plt.subplots()

output["mass"]["ee", :].plot1d(ax=ax, label=r"$ee$")
output["mass"]["ee_same_sign", :].plot1d(ax=ax, label=r"$ee$ same sign")
ax.legend()

fig.savefig(plot_dir / "ee_mass.png")

## Preview: upcoming coffea features

Having built analysis results by hand, here is a preview of upcoming coffea machinery for describing datasets and scaling the work that produces those results.

The cells below preview features that are **not yet released** — they live in open *draft* pull requests against the coffea repository. Each demo is **guarded**: it detects whether the feature is present (by import / signature introspection, never by version number, since these are unreleased branches) and prints a gentle note instead of raising if it is missing. This section is therefore safe to "Run All" in the default environment.

To actually exercise the demos, launch one of the preview environments defined in `pixi.toml`:

```bash
pixi run -e preview jupyter lab           # pydantic dataset-tools extensions: PRs #1579, #1600, #1601
pixi run -e preview-compute jupyter lab    # the coffea.compute execution refactor: PR #1470
```

Those environments install coffea straight from the PR branches, so the exact API may drift before release.

In [ ]:
# --- Feature detection for the preview demos below ---------------------------
# These upcoming features live in *unreleased* draft PRs, so we never rely on a
# version number; each is detected by import / signature introspection. When a
# feature is absent the demo cells print a gentle note instead of raising, so
# this whole section is safe to "Run All" in the default environment.
import importlib.util
import inspect
from pathlib import Path

import coffea
from coffea.dataset_tools import preprocess as _preprocess


def _has_param(func, name):
    try:
        return name in inspect.signature(func).parameters
    except (TypeError, ValueError):
        return False


HAS_PP_BACKENDS = _has_param(_preprocess, "backend")             # draft PR #1579
HAS_PP_METADATA = _has_param(_preprocess, "metadata_extractor")  # draft PR #1600
HAS_MUTABLE_STEPS = importlib.util.find_spec("coffea.dataset_tools.mutable_steps") is not None  # draft PR #1601
HAS_COMPUTE = importlib.util.find_spec("coffea.compute") is not None  # draft PR #1470


def preview_note(feature, pr):
    print(
        f"[preview] '{feature}' is not available in this coffea build ({coffea.__version__}).\n"
        f"          It ships in draft PR {pr}. To try it, launch a preview environment:\n"
        f"            pixi run -e preview jupyter lab           # pydantic dataset-tools extensions (#1579/#1600/#1601)\n"
        f"            pixi run -e preview-compute jupyter lab    # coffea.compute execution refactor (#1470)"
    )


# A small, network-free sample so the preview demos run wherever this repo is
# checked out (they fall back gracefully if it is missing).
_preview_file = Path("../columnar/data/SMHiggsToZZTo4L.root")
_preview_fileset = {
    "demo": {"files": {str(_preview_file): "Events"}, "metadata": {"xsec": 1.0}}
}

print(f"coffea {coffea.__version__}")
for _flag in ["HAS_PP_BACKENDS", "HAS_PP_METADATA", "HAS_MUTABLE_STEPS", "HAS_COMPUTE"]:
    print(f"  {_flag} = {globals()[_flag]}")

### 1. Pydantic dataset specifications

*Released in coffea 2026.7 (PR #1528) — the foundation the previews build on.*

Filesets can now be expressed as validated `pydantic` models (`DataGroupSpec` / `DatasetSpec` / `ROOTFileSpec`, ...). Malformed filesets fail fast with clear errors, and the models carry form and metadata around for the tools below.

In [ ]:
# 1. Pydantic dataset specifications (released in coffea 2026.7, PR #1528)
# The classic "dict-in / dict-out" fileset still works, but datasets can now be
# expressed as *validated* pydantic models, catching malformed filesets early.
from coffea.dataset_tools import ModelFactory, DatasetSpec

spec = ModelFactory.dict_to_datasetspec(_preview_fileset["demo"])
print("type:", type(spec).__name__, "| is DatasetSpec:", isinstance(spec, DatasetSpec))
print("validated metadata:", dict(spec.metadata))
print("file specs:", [type(fs).__name__ for fs in spec.files.values()])
# round-trip back to a plain dict when a legacy API needs one
_roundtrip = ModelFactory.datasetspec_to_dict(spec)

### 2. Non-dask preprocessing backends — draft PR #1579

Released `preprocess()` builds a **dask-awkward** graph to discover file chunks. PR #1579 adds a `backend=` switch (`"iterative"`, `"futures"`, `"dask"`) so preprocessing can run with **no dask dependency**, plus a dedicated `preprocess_rntuple()` for RNTuple inputs. The backend classes (`IterativeBackend`, `FuturesBackend`, ...) are explicitly designed to plug into the `coffea.compute` refactor below.

In [ ]:
# 2. Non-dask preprocessing backends  (draft PR #1579)
from coffea.dataset_tools import preprocess

if HAS_PP_BACKENDS and _preview_file.exists():
    available, report = preprocess(
        _preview_fileset,
        step_size=50_000,
        save_form=False,
        backend="iterative",  # or "futures"; "dask" reproduces the legacy path
        skip_bad_files=True,
    )
    finfo = list(available["demo"]["files"].values())[0]
    print("preprocessed with the dask-free 'iterative' backend")
    print("  steps discovered:", finfo["steps"])
    print("  num_entries:", finfo["num_entries"])
elif not HAS_PP_BACKENDS:
    preview_note("preprocess(backend=...)", "#1579")
else:
    print("[preview] sample file not found; skipping the live run.")

### 3. User-supplied metadata extraction — draft PR #1600

Computing per-dataset quantities such as the sum of generator weights normally means an extra pass over the files. PR #1600 adds `metadata_extractor` (called once per file on the open handle) and `metadata_reducer` (called once per dataset) hooks to `preprocess()`, folding that work into the preprocessing pass.

In [ ]:
# 3. User-supplied metadata extraction during preprocessing  (draft PR #1600)
from coffea.dataset_tools import preprocess

if HAS_PP_METADATA and _preview_file.exists():
    def per_file(file_handle):
        # runs once per file, on the open uproot file handle
        return {"nentries": int(file_handle["Events"].num_entries)}

    def per_dataset(per_file_meta):
        # reduce the per-file dicts into dataset-level metadata
        return {"nentries_total": sum(m["nentries"] for m in per_file_meta.values())}

    available, _ = preprocess(
        _preview_fileset,
        step_size=50_000,
        save_form=False,
        backend="iterative",
        metadata_extractor=per_file,
        metadata_reducer=per_dataset,
        skip_bad_files=True,
    )
    print("dataset metadata after extraction:", dict(available["demo"]["metadata"]))
elif not HAS_PP_METADATA:
    preview_note("preprocess(metadata_extractor=..., metadata_reducer=...)", "#1600")
else:
    print("[preview] sample file not found; skipping the live run.")

### 4. Adaptive / resizable steps — draft PR #1601

Fixed step sizes over- or under-shoot when chunk cost varies. This **prototype** adds a resizable step generator whose size can be renegotiated mid-stream through the generator `.send()` channel (the same channel `coffea.compute`'s `Computable.gen_steps` uses), plus a `run_adaptive_steps` driver governed by a `WallTimeStepPolicy`. The API is explicitly marked unstable.

In [ ]:
# 4. Adaptive / resizable steps  (draft PR #1601, prototype -- API may change)
if HAS_MUTABLE_STEPS:
    from coffea.dataset_tools.mutable_steps import resizable_steps

    gen = resizable_steps(0, 1_000, 200)
    produced = [next(gen)]
    try:
        while True:
            # after the first chunk, ask the generator to shrink the step to 100
            produced.append(gen.send(100))
    except StopIteration:
        pass
    print("resizable_steps, shrunk mid-stream via .send(100):")
    print(" ", produced)

    # Higher-level driver, operating on a preprocessed pydantic DatasetSpec:
    print(
        "\nHigher-level API (illustrative):\n"
        "    from coffea.dataset_tools.mutable_steps import (\n"
        "        iter_dataset_steps, run_adaptive_steps, WallTimeStepPolicy)\n"
        "    policy = WallTimeStepPolicy(target_seconds=30)\n"
        "    total = run_adaptive_steps(dataset_spec, work_fn, step_size=100_000, policy=policy)"
    )
else:
    preview_note("coffea.dataset_tools.mutable_steps", "#1601")

### 5. A unified execution protocol: `coffea.compute` — draft PR #1470

The largest change on the horizon. PR #1470 introduces `coffea.compute`, replacing the `Processor` / `Executor` / `Runner` trio with a single `Backend` **protocol**. Work is expressed as a `Computable` (a `Dataset` mapped through a function via `.map_steps`), handed to any backend's `.compute()`, which returns a non-blocking `Task` exposing `.result()`, `.partial_result()`, `.wait()`, and `.cancel()`. The preprocessing backends (#1579) and resizable steps (#1601) are stepping stones toward this unified interface. The one-liner it enables:

```python
with ThreadedBackend() as backend:
    total = backend.compute(dataset.map_steps(process)).result()
```

This backend is a genuine work in progress, so the demo below is guarded to show the protocol *shape* even where it does not yet fully execute.

In [ ]:
# 5. A unified execution protocol: coffea.compute  (draft PR #1470, WIP)
# PR #1470 replaces the Processor/Executor/Runner trio with a single `Backend`
# protocol. A `Computable` (a Dataset mapped through a function via .map_steps)
# is handed to any backend's .compute(), returning a non-blocking Task with
# .result()/.partial_result(). This is what tutorial scaleout could look like
# once the refactor lands:
if HAS_COMPUTE:
    from coffea.compute.data import Dataset, File, ContextDataset
    from coffea.compute.backends.threaded import ThreadedBackend

    dataset = Dataset(
        files=[File(path=str(_preview_file), steps=[(0, 50_000), (50_000, 100_000)])],
        metadata=ContextDataset(dataset_name="demo", cross_section=None),
    )

    def count(events):  # a plain callable *is* the processor
        return len(events)

    computable = dataset.map_steps(count)
    print(f"built a Computable with {len(computable)} work element(s)")
    print(
        "the target one-liner:\n"
        "    with ThreadedBackend() as backend:\n"
        "        total = backend.compute(computable).result()"
    )
    try:
        with ThreadedBackend() as backend:
            total = backend.compute(computable).result()
        print("result:", total)
    except Exception as exc:  # coffea.compute is a work-in-progress preview
        print(
            f"[preview] coffea.compute did not execute here yet ({type(exc).__name__}); "
            "the protocol shape above is the point of this WIP preview."
        )
else:
    preview_note("coffea.compute", "#1470")